In [4]:
import pandas as pd
from pprint import pprint
import json
import re
from typing import Dict, List, Any


In [12]:
ds_plannerStudy_628 = pd.read_json(
    "Original datasets/plannerStudy_628.json", lines=True
)

In [13]:
samples=[]
for row in range(1):
    samples.append(
        {col: ds_plannerStudy_628[col][row] for col in ds_plannerStudy_628.columns}
    )
pprint(samples[0])

{'input': {'additionalnotes': 'None',
           'busylevels': "{'monday': 'really busy', 'tuesday': 'very busy', "
                         "'wednesday': 'very busy', 'thursday': 'a little "
                         "busy', 'friday': 'very busy', 'saturday': 'busy', "
                         "'sunday': 'busy'}",
           'course': 'AP Environmental Science',
           'date': "{'today': 'Saturday', 'date': datetime.date(2022, 9, 6)}",
           'difficulty': 'hard',
           'effort': 'low effort',
           'pace': 'fast pace',
           'timeuntiltest': '7 day(s)',
           'unit': 'The Impact of Microplastic Pollution on Marine '
                   'Biodiversity and Ecosystem Health',
           'username': 'Jessica Ali'},
 'instruction': 'You are a study planner assistant in an educational app. Your '
                'task is to offer detailed steps specifically designed for the '
                'course, focusing on the unit/test. DO NOT give more than 10 '
           

In [ ]:
import json
import pandas as pd


def convert_row_to_schema(row):
    """
    Convert a single row of study plan data into the target schema.
    Assumes row contains keys like: course, unit, timeuntiltest, difficulty, busylevels, etc.
    """
    # Build baseline constraints
    baseline = {
        "fixed_constraints": f"Test scheduled in {row.get('timeuntiltest', 'N/A')}",
        "physical_constraints": f"Busy schedule: {row.get('busylevels', 'N/A')}",
        "non_negotiables": f"Must cover all unit topics for {row.get('unit', '')}",
    }

    # Build tasks from sections (if provided)
    tasks = []
    if "sections" in row:
        for i, section in enumerate(row["sections"]):
            tasks.append(
                {
                    "index": i,
                    "task": section.get("name", ""),
                    "description": section.get("outline", ""),
                    "is_repeatable": False,
                    "repeat_pattern": "none",
                    "estimated_duration_minutes": {
                        "min": 45,  # default estimate
                        "max": 90,  # default estimate
                    },
                }
            )

    schema = {
        "goal": f"Prepare for {row.get('course', '')} test on {row.get('unit', '')}",
        "goal_category": "one_time",
        "goal_type": "academic study",
        "time_horizon": row.get("timeuntiltest", ""),
        "description": f"A study plan for {row.get('course', '')} focusing on {row.get('unit', '')}.",
        "baseline": baseline,
        "tasks": tasks,
    }

    return schema


# Example: converting a dataset of 1k rows
def convert_dataset(input_file, output_file):
    """
    Reads a JSON or CSV dataset of study plans and converts each row.
    """
    # Load dataset (assuming JSON lines or CSV)
    if input_file.endswith(".json"):
        with open(input_file, "r") as f:
            data = json.load(f)
    else:
        df = pd.read_csv(input_file)
        data = df.to_dict(orient="records")

    # Convert each row
    converted = [convert_row_to_schema(row) for row in data]

    # Save output
    with open(output_file, "w") as f:
        json.dump(converted, f, indent=2)


# Example usage:
# convert_dataset("study_plans.json", "converted_schema.json")
# convert_dataset("study_plans.csv", "converted_schema.json")